In [4]:
import io
import zipfile
import urllib.request
import numpy as np
import ssl
import pandas as pd
from gensim.models import Word2Vec

In [7]:
ssl_context = ssl._create_unverified_context()

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

response = urllib.request.urlopen(url, context=ssl_context)
archive = zipfile.ZipFile(io.BytesIO(response.read()))

ratings = pd.read_csv(archive.open("ml-latest-small/ratings.csv"))
movies = pd.read_csv(archive.open("ml-latest-small/movies.csv"))

id_to_title = dict(zip(movies["movieId"].astype(str), movies["title"]))
title_to_id = dict(zip(movies["title"], movies["movieId"].astype(str)))

In [8]:
positive_ratings = ratings[ratings["rating"] >= 4.0].copy()

user_movies = (
    positive_ratings
    .groupby("userId")["movieId"]
    .apply(lambda x: [str(m_id) for m_id in x])
    .tolist()
)

In [9]:
model = Word2Vec(
    sentences=user_movies,
    vector_size=50,
    window=5,
    min_count=2,
    sg=1,
    workers=4,
    epochs=30,
    seed=42
)

In [10]:
target_title = "Toy Story (1995)"
target_id = title_to_id[target_title]

similar_movies = model.wv.most_similar(target_id, topn=10)

for sim_id, similarity in similar_movies:
    movie_name = id_to_title.get(sim_id, sim_id)
    print(f"{movie_name:50s} | similarity: {similarity:.3f}")

Dracula: Dead and Loving It (1995)                 | similarity: 0.891
GoldenEye (1995)                                   | similarity: 0.889
American President, The (1995)                     | similarity: 0.889
Casino (1995)                                      | similarity: 0.889
Father of the Bride Part II (1995)                 | similarity: 0.886
Leaving Las Vegas (1995)                           | similarity: 0.882
Heat (1995)                                        | similarity: 0.881
Ace Ventura: When Nature Calls (1995)              | similarity: 0.880
Grumpier Old Men (1995)                            | similarity: 0.877
Sense and Sensibility (1995)                       | similarity: 0.877


In [11]:
liked_titles = [
    "Toy Story (1995)",
    "Finding Nemo (2003)",
    "Shrek (2001)"
]

liked_ids = [title_to_id[t] for t in liked_titles if t in title_to_id and title_to_id[t] in model.wv]

user_vectors = [model.wv[m_id] for m_id in liked_ids]
user_vector = np.mean(user_vectors, axis=0)

recommendations = model.wv.similar_by_vector(user_vector, topn=15)

shown = 0
for sim_id, similarity in recommendations:
    if sim_id in liked_ids:
        continue
    movie_name = id_to_title.get(sim_id, sim_id)
    print(f"  - {movie_name:45s} (score: {similarity:.3f})")
    shown += 1
    if shown == 5:
        break

  - Legally Blonde (2001)                         (score: 0.785)
  - A.I. Artificial Intelligence (2001)           (score: 0.781)
  - Moulin Rouge (2001)                           (score: 0.773)
  - Bruce Almighty (2003)                         (score: 0.757)
  - Ghost Ship (2002)                             (score: 0.757)


In [12]:
model_window_2 = Word2Vec(
    sentences=user_movies,
    vector_size=50,
    window=2,
    min_count=2,
    sg=1,
    epochs=30,
    seed=42
)

top_w5 = model.wv.most_similar(target_id, topn=5)
top_w2 = model_window_2.wv.most_similar(target_id, topn=5)

print(f"{'Модель (window=5)':35s} | {'Модель (window=2)':35s}")
print("-" * 75)

for (id_5, sim_5), (id_2, sim_2) in zip(top_w5, top_w2):
    name_5 = id_to_title.get(id_5, id_5)
    name_2 = id_to_title.get(id_2, id_2)
    print(f"{name_5[:33]:33s} ({sim_5:.2f}) | {name_2[:33]:33s} ({sim_2:.2f})")

Модель (window=5)                   | Модель (window=2)                  
---------------------------------------------------------------------------
Dracula: Dead and Loving It (1995 (0.89) | GoldenEye (1995)                  (0.95)
GoldenEye (1995)                  (0.89) | Casino (1995)                     (0.95)
American President, The (1995)    (0.89) | Father of the Bride Part II (1995 (0.94)
Casino (1995)                     (0.89) | Heat (1995)                       (0.93)
Father of the Bride Part II (1995 (0.89) | Dracula: Dead and Loving It (1995 (0.92)
